# Merge: full daily corona panel + reduced census -> Corona_Mifkad2_merged.csv

Left-joins the full daily corona panel (`Corona_loc_processed.csv`, built by
`preprocessing_corona_porat.ipynb` - one row per area per date, unchanged
and NOT modified by this notebook) with the dimension-reduced census
(`Mifkad_2_reduced.csv`, 27 columns, built by
`preprocessing_mifkak2_naama.ipynb`) on `City_agas_code`.

**Saved to a new file, `Corona_Mifkad2_merged.csv`** - this does not exist
yet and nothing else is overwritten. This exact filename/path is already
`Merged_df_PATH` in `EDA.ipynb`, read there as `df_merged`.

**Columns required, read directly off `EDA.ipynb`'s own cells** (not
guessed): `df_merged` is indexed by `City_agas_code` and used with `date`,
`town`, `infection_rate`, `percent_vaccinated_first_dose`,
`percent_vaccinated_second_dose`, `percent_vaccinated_third_dose`, and
`panic_index` (EDA's own comment: "daily tests per 1,000 residents"). The
first four already exist on the corona side; the vaccination percentages
and `panic_index` need a population denominator, which only the census side
has (`pop_approx`) - hence why this merge exists. EDA.ipynb itself is not
touched or run by this notebook.


In [ ]:
from pathlib import Path
import pandas as pd

CORONA_PATH = Path("/home/bcrlab/igguest/porat_naama/data/processed/Corona_loc_processed.csv")
MIFKAD_PATH = Path("/home/bcrlab/igguest/porat_naama/data/processed/mifkad/Mifkad_2_reduced.csv")
OUT_PATH = Path("/home/bcrlab/igguest/porat_naama/data/processed/Corona_Mifkad2_merged.csv")

assert not OUT_PATH.exists(), f"{OUT_PATH} already exists - not overwriting, stopping."

output_log = []

def log_output(path, description, description_he):
    output_log.append({"file_path": str(path), "contents": description, "contents_he": description_he})
    print("Saved:", path)


## Load both sides

`df_corona` is the full daily panel, read as-is (no changes made to the
source file itself). `df_mifkad` is the full 27-column dimension-reduced
census table.


In [ ]:
df_corona = pd.read_csv(CORONA_PATH, dtype={"City_agas_code": str}, parse_dates=["date"])
df_mifkad = pd.read_csv(MIFKAD_PATH, dtype={"City_agas_code": str})

assert df_mifkad["City_agas_code"].is_unique, "Mifkad key is not unique - merge would explode rows"

print(f"corona (daily panel): {df_corona.shape[0]:,} rows, {df_corona['City_agas_code'].nunique():,} areas, "
      f"{df_corona['date'].min().date()} to {df_corona['date'].max().date()}")
print(f"mifkad (reduced): {df_mifkad.shape[0]:,} areas, {df_mifkad.shape[1]} cols")


## Merge

**Left join** on `City_agas_code`: every daily-panel row is kept, all 27
reduced census columns are broadcast onto it where a match exists.
Unmatched areas keep NaN census columns, flagged via `has_census` rather
than silently dropped - same pattern as `merge_corona_mifkad2.ipynb` (the
wave-2 modeling merge).


In [ ]:
df_merged = df_corona.merge(
    df_mifkad,
    on="City_agas_code",
    how="left",
    validate="many_to_one",
)

df_merged["has_census"] = df_merged["pop_approx"].notna().astype(int)

n_rows = len(df_merged)
n_areas = df_merged["City_agas_code"].nunique()
n_areas_with_census = df_merged.loc[df_merged["has_census"] == 1, "City_agas_code"].nunique()
print(f"Rows: {n_rows:,}")
print(f"Areas: {n_areas:,} total, {n_areas_with_census:,} with a census match "
      f"({100 * n_areas_with_census / n_areas:.1f}%)")


## Columns EDA.ipynb needs but neither source table has

`EDA.ipynb` plots `percent_vaccinated_first/second/third_dose` (the corona
side only has cumulative counts) and `panic_index`, its own name for daily
tests per 1,000 residents. Both need `pop_approx` as the denominator, so
they can only be computed here, after the merge. Rows with no census match
(`has_census == 0`) come out NaN rather than using a wrong denominator.


In [ ]:
dose_cols = {
    "first": "accumulated_vaccination_first_dose",
    "second": "accumulated_vaccination_second_dose",
    "third": "accumulated_vaccination_third_dose",
}
for dose_name, col in dose_cols.items():
    df_merged[f"percent_vaccinated_{dose_name}_dose"] = 100 * df_merged[col] / df_merged["pop_approx"]

df_merged["panic_index"] = 1000 * df_merged["tested_daily"] / df_merged["pop_approx"]

df_merged[[f"percent_vaccinated_{d}_dose" for d in dose_cols] + ["panic_index"]].describe()


## Drop columns superseded by the derived columns above

`accumulated_vaccination_first/second/third_dose` and `tested_daily` are
now redundant with `percent_vaccinated_*_dose` and `panic_index`;
`infection_rate_window_days` is an internal diagnostic from the adaptive
infection-rate calculation in `preprocessing_corona_porat.ipynb`, not used
by `EDA.ipynb`. Dropped only from this merged output - the source files
(`Corona_loc_processed.csv`) are untouched.


In [ ]:
cols_to_drop = [
    "accumulated_vaccination_first_dose",
    "accumulated_vaccination_second_dose",
    "accumulated_vaccination_third_dose",
    "infection_rate_window_days",
    "tested_daily",
]
df_merged = df_merged.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns. Remaining: {df_merged.shape[1]} cols")


## Save

New file, `Corona_Mifkad2_merged.csv` - `Merged_df_PATH` in `EDA.ipynb`.
`City_agas_code` is kept as a plain column (not the index); `EDA.ipynb`'s
own load cell already does `set_index("City_agas_code")` after reading it.


In [ ]:
df_merged.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

log_output(
    OUT_PATH,
    "Full daily corona panel (Corona_loc_processed.csv) left-joined with the reduced census "
    "(Mifkad_2_reduced.csv, 27 columns) on City_agas_code. One row per area per date. Adds "
    "has_census, percent_vaccinated_first/second/third_dose (from the existing "
    "accumulated_vaccination_*_dose columns / pop_approx), and panic_index (tested_daily per "
    "1,000 residents / pop_approx). Areas with no census match keep has_census=0 and NaN "
    "census/derived columns instead of being dropped. This is Merged_df_PATH in EDA.ipynb.",
    "פאנל הקורונה היומי המלא (Corona_loc_processed.csv) ממוזג (left join) עם המפקד המצומצם "
    "(Mifkad_2_reduced.csv, 27 עמודות) לפי City_agas_code. שורה אחת לכל אזור לכל תאריך. "
    "מתווספות has_census, percent_vaccinated_first/second/third_dose (מעמודות "
    "accumulated_vaccination_*_dose הקיימות חלקי pop_approx), ו-panic_index (tested_daily "
    "ל-1,000 תושבים חלקי pop_approx). אזורים ללא התאמת מפקד נשארים עם has_census=0 ו-NaN "
    "בעמודות המפקד/הנגזרות במקום להיזרק. זהו Merged_df_PATH ב-EDA.ipynb.",
)

manifest_path = OUT_PATH.parent / "merge_full_panel_output_manifest.csv"
pd.DataFrame(output_log).to_csv(manifest_path, index=False, encoding="utf-8-sig")
print("Saved:", manifest_path)
